# Fraud Prediction Consumer

This notebook listens to the Kafka topic:

fraud-predictions

Every prediction published by Notebook 18 is displayed in real time.

This notebook represents downstream systems such as dashboards, databases and alert services.

## Imports

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Create Spark Session

In [12]:
spark = (
    SparkSession.builder
    .appName("FraudPredictionConsumer")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

## Stop Existing Streams

In [13]:
for stream in spark.streams.active:
    stream.stop()

print("Stopped existing streams.")

Stopped existing streams.


## Read Kafka Topic

In [14]:
stream_df = (

    spark.readStream

    .format("kafka")

    .option(
        "kafka.bootstrap.servers",
        "localhost:9092"
    )

    .option(
        "subscribe",
        "fraud-predictions"
    )

    .option(
        "startingOffsets",
        "latest"
    )

    .load()

)

## Prediction Schema

In [15]:
schema = StructType([

    StructField("transaction_id", StringType()),

    StructField("event_timestamp", StringType()),

    StructField("Amount", DoubleType()),

    StructField("Prediction", IntegerType()),

    StructField("Fraud_Probability", DoubleType()),

    StructField("Risk_Level", StringType()),

    StructField("Status", StringType())

])

## Parse JSON

In [16]:
parsed_df = (

    stream_df

    .selectExpr("CAST(value AS STRING)")

    .select(

        from_json(
            col("value"),
            schema
        ).alias("prediction")

    )

    .select("prediction.*")

)

## Verify Schema

In [17]:
parsed_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Prediction: integer (nullable = true)
 |-- Fraud_Probability: double (nullable = true)
 |-- Risk_Level: string (nullable = true)
 |-- Status: string (nullable = true)



## Prediction Consumer Function

In [18]:
def display_predictions(batch_df, batch_id):

    pdf = batch_df.toPandas()

    if pdf.empty:
        return

    print("\n")
    print("=" * 120)
    print(f"Prediction Batch {batch_id}")
    print("=" * 120)

    print(
        pdf[
            [
                "transaction_id",
                "Amount",
                "Fraud_Probability",
                "Risk_Level",
                "Status"
            ]
        ]
    )

    frauds = pdf[pdf["Prediction"] == 1]

    if len(frauds):

        print("\n")
        print("🚨 FRAUD ALERT 🚨")
        print("=" * 120)

        print(
            frauds[
                [
                    "transaction_id",
                    "Amount",
                    "Fraud_Probability",
                    "Risk_Level"
                ]
            ]
        )

    print("\nReceived", len(pdf), "prediction(s)")

## Start Consumer

In [19]:
query = (

    parsed_df

    .writeStream

    .foreachBatch(display_predictions)

    .outputMode("append")

    .option(
        "checkpointLocation",
        "../checkpoints/prediction_consumer"
    )

    .start()

)

## Keep Consumer Alive

In [20]:
query.awaitTermination()



Prediction Batch 229
                         transaction_id  Amount  Fraud_Probability Risk_Level  \
0  1fd33baa-05a9-462a-ba9b-ab60f21a2d48   21.55                0.0        Low   

    Status  
0  Genuine  

Received 1 prediction(s)


Prediction Batch 230
                         transaction_id  Amount  Fraud_Probability Risk_Level  \
0  4e867d8c-758a-4f72-8b86-f3a24e8f51e7    59.9                0.0        Low   

    Status  
0  Genuine  

Received 1 prediction(s)


Prediction Batch 231
                         transaction_id  Amount  Fraud_Probability Risk_Level  \
0  5ebf9974-129b-42e2-878c-f49d0125d2a4    0.75                0.0        Low   

    Status  
0  Genuine  

Received 1 prediction(s)


Prediction Batch 232
                         transaction_id  Amount  Fraud_Probability Risk_Level  \
0  36f7e040-7f26-410b-a3b8-4057ce4e174d   45.71                0.0        Low   

    Status  
0  Genuine  

Received 1 prediction(s)


Prediction Batch 233
                         

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/yaseen/Data_Analysis/Project_Files/Financial_Fraud_Detection/venv/lib/python3.13/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 